# Family A - ConvNeXtV2-tiny launcher (Colab / Kaggle)

Runs `families/A_convnextv2/train.py` from the repo; the same script runs on the Mac.

**Colab**: Runtime > Change runtime type > L4 or A100. Add the secret `KAGGLE_API_TOKEN`
(key icon, notebook access on). Outputs go to Google Drive (`MyDrive/an2dl/outputs`), so a
disconnect loses nothing: re-run the notebook and finished folds are skipped.

**Kaggle**: Settings > Accelerator > GPU T4 x2, Internet on. Add the competition with
"Add Input". For long runs use *Save Version > Save & Run All* (runs in background, up to 12 h);
outputs land in `/kaggle/working/outputs`.

In [ ]:
# ---- what to run ----
BRANCH = 'davide/model'
ARGS = '--folds 0 --no-test'   # quick experiment; '' = full 5 folds + test + submission
RUN_NAME = ''                  # '' = the one in config.yaml
CHANGE = ''                    # one line for runs/<run>.json
EXTRA = ''                     # e.g. '--set color_mode=gray --epochs 15'

In [ ]:
import os, subprocess, sys
ON_KAGGLE = os.path.isdir('/kaggle/input')
ON_COLAB = 'google.colab' in sys.modules
REPO = '/tmp/repo' if ON_KAGGLE else '/content/repo'  # not saved as Kaggle output

if ON_COLAB:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')  # read by train.py, never printed

if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH,
                    'https://github.com/davidemeta/ANN-DL-Challenge.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
!pip install -q "timm>=1.0.20" kagglehub pyyaml
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
cmd = f'python {REPO}/families/A_convnextv2/train.py {ARGS} {EXTRA}'
if RUN_NAME:
    cmd += f' --run-name {RUN_NAME}'
if CHANGE:
    cmd += f' --change "{CHANGE}"'
print(cmd)
!{cmd}

In [ ]:
import glob, json
for path in sorted(glob.glob(f'{REPO}/runs/A_*.json')):
    r = json.load(open(path))
    print(r['run'], '| folds', r['folds_done'], '| OOF f1_clean', r['cv_oof'],
          '| f1_shifted', r['cv_shifted_folds'], '|', r.get('submission', ''))

`runs/<run>.json` is written inside the cloned repo: copy it back to your machine (it is
also in the run's output folder) and commit it from there.